# Μοντελοποίηση Βαθμολογιών Εμπειρίας Ασθενών σε Εγκαταστάσεις και Ειδικότητες με την PROC FACTMAC

## Σύνοψη για Στελέχη

Ένα πολυκεντρικό σύστημα υγείας θέλει να μάθει τη **δομή αλληλεπίδρασης** μεταξύ εγκαταστάσεων περίθαλψης και κλινικών ειδικοτήτων που καθορίζει τις βαθμολογίες ικανοποίησης ασθενών, ώστε να εντοπίζει συνδυασμούς εγκατάστασης/ειδικότητας που υπο- ή υπερ-αποδίδουν. Αυτό το σημειωματάριο προσαρμόζει μια μηχανή παραγοντοποίησης με την **PROC FACTMAC**, αντιμετωπίζοντας τα `facility` και `specialty` ως τα δύο ονομαστικά πεδία χαρακτηριστικών και τη βαθμολογία ικανοποίησης 1-10 ως τον στόχο διαστήματος, και στη συνέχεια αξιολογεί τις ανακατασκευασμένες βαθμολογίες.

Στις 100 προσομοιωμένες επισκέψεις το μοντέλο φτάνει σε **R-Square εκπαίδευσης 0.516** (μέσο απόλυτο σφάλμα 0.95 μονάδες βαθμολογίας, RASE 1.25), και οι προβλεπόμενοι μέσοι όροι ανά κελί διαχωρίζουν σαφώς τους ισχυρότερους και ασθενέστερους συνδυασμούς — `CentralMed`/`Cardiology` στην κορυφή έναντι `SouthClinic`/`Cardiology` στον πάτο — ανακτώντας την ενσωματωμένη αλληλεπίδραση αντί να συρρικνώνεται στον γενικό μέσο όρο περίπου 6.8.

## Πηγές Δεδομένων

Όλα τα δεδομένα παράγονται εσωτερικά από ένα βήμα DATA (`call streaminit(20240531)` + `rand()`), οπότε το σημειωματάριο είναι πλήρως αυτόνομο — χωρίς εξωτερικά αρχεία ή πρόσβαση δικτύου. Αυτό το περιβάλλον εκτελείται χωρίς άδεια, κάτι που περιορίζει την έξοδο σε 100 παρατηρήσεις, οπότε ο σχεδιασμός έχει μέγεθος για να επιδείξει τη μηχανή παραγοντοποίησης **εντός 100 επισκέψεων**: τρεις εγκαταστάσεις σε συνδυασμό με δύο ειδικότητες (έξι κελιά, με μέσο όρο περίπου 17 επισκέψεις το καθένα — αρκετό σήμα ανά κελί ώστε η στοχαστική κάθοδος κλίσης να ανακτήσει τη δομή).

**WORK.ENCOUNTERS** — 100 συνθετικές επισκέψεις ασθενών (μία γραμμή ανά επίσκεψη).

| Μεταβλητή | Τύπος | Ρόλος | Περιγραφή |
|----------|------|------|-------------|
| `facility` | char(12) | Είσοδος (ονομαστική) | Χώρος περίθαλψης — `NorthClinic`, `CentralMed`, ή `SouthClinic` |
| `specialty` | char(14) | Είσοδος (ονομαστική) | Κλινική ειδικότητα — `Cardiology` ή `Oncology` |
| `satisfaction` | num | Στόχος (διαστήματος) | Βαθμολογία εμπειρίας ασθενή σε κλίμακα 1-10, που παράγεται από πόλωση εγκατάστασης + πόλωση ειδικότητας + έναν γνήσιο όρο αλληλεπίδρασης εγκατάστασης×ειδικότητας + θόρυβο Gauss, στη συνέχεια περικόπτεται στο [1,10] και στρογγυλοποιείται στο 0.1 |

Ο λανθάνων σχεδιασμός ενσωματώνει καλά διαχωρισμένες πολώσεις ανά εγκατάσταση και ανά ειδικότητα συν ένα ισχυρό εφέ αλληλεπίδρασης, οπότε μια μηχανή παραγοντοποίησης θα πρέπει να ανακτήσει δομή που ένας μέσος όρος μόνο-εγκατάστασης ή μόνο-ειδικότητας θα έχανε.

# Μοντελοποίηση Βαθμολογιών Εμπειρίας Ασθενών με την PROC FACTMAC

Τα πολυκεντρικά συστήματα υγείας συλλέγουν βαθμολογίες ικανοποίησης ασθενών σε πολλές **εγκαταστάσεις** και κλινικές **ειδικότητες**. Οι μέσες βαθμολογίες ανά εγκατάσταση ή ανά ειδικότητα μόνο κρύβουν το ενδιαφέρον σήμα: μια ειδικότητα μπορεί να διαπρέπει σε μια εγκατάσταση και να δυσκολεύεται σε μια άλλη. Μια **μηχανή παραγοντοποίησης** συλλαμβάνει ακριβώς αυτό το είδος ζευγαρωτής αλληλεπίδρασης μαθαίνοντας λανθάνοντες παράγοντες για κάθε εγκατάσταση και κάθε ειδικότητα, και στη συνέχεια μοντελοποιώντας κάθε βαθμολογία ως γενικό μέσο όρο συν επιδράσεις ανά χαρακτηριστικό συν την αλληλεπίδρασή τους.

Η `PROC FACTMAC` προσαρμόζει αυτό το μοντέλο με στοχαστική κάθοδο κλίσης. Σε αυτό το σημειωματάριο:

1. Δημιουργούμε έναν συνθετικό πίνακα επισκέψεων με ενσωματωμένη αλληλεπίδραση εγκατάστασης x ειδικότητας, με μέγεθος κατάλληλο για το περιβάλλον 100 παρατηρήσεων.
2. Προσαρμόζουμε μια μηχανή παραγοντοποίησης με την `PROC FACTMAC`, ζητώντας προβλέψεις βαθμολόγησης και ιστορικό επαναλήψεων.
3. Αξιολογούμε το σφάλμα ανακατασκευής και αναδεικνύουμε τους συνδυασμούς εγκατάστασης x ειδικότητας που το μοντέλο επισημαίνει ως ισχυρότερους και ασθενέστερους.

## Βήμα 1 - Δημιουργία συνθετικών δεδομένων εμπειρίας ασθενών

Προσομοιώνουμε 100 επισκέψεις σε 3 εγκαταστάσεις και 2 ειδικότητες. Κάθε εγκατάσταση και ειδικότητα φέρει μια κρυφή πόλωση, και προσθέτουμε έναν γνήσιο όρο **αλληλεπίδρασης** ώστε ορισμένοι συνδυασμοί εγκατάστασης/ειδικότητας να βαθμολογούνται υψηλότερα ή χαμηλότερα από ό,τι θα προέβλεπαν οι μεμονωμένες πολώσεις τους. Ο θόρυβος Gauss (τυπική απόκλιση 0.25) κάνει τις βαθμολογίες ρεαλιστικές, και περικόπτουμε στην κλίμακα 1-10 και στρογγυλοποιούμε σε ένα δεκαδικό ψηφίο. Ο σπόρος `call streaminit` καθιστά τα δεδομένα αναπαραγώγιμα.

In [1]:
ΔΕΔΟΜΕΝΑ encounters;
    CALL streaminit(20240531);
    LENGTH facility $12 specialty $14;

    /* Κρυφές πολώσεις βαθμολόγησης ανά εγκατάσταση και ειδικότητα.  */
    /* Οι τιμές facility/specialty παραμένουν ASCII (NorthClinic,    */
    /* CentralMed, SouthClinic, Cardiology, Oncology): η PROC        */
    /* FACTMAC κωδικοποιεί εσωτερικά τις ονομαστικές κατηγορίες και  */
    /* πολυ-byte τιμές αλλοιώνουν την προσαρμογή του μοντέλου· η     */
    /* ελληνική ετικέτα στήλης δίνεται παρακάτω μέσω LABEL.          */
    ARRAY f_bias[3] _temporary_ (1.5 0.0 -1.5);
    ARRAY s_bias[2] _temporary_ (1.0 -1.0);

    ΕΠΑΝΑΛΗΨΗ enc = 1 ΕΩΣ 100;
        fi = 1 + floor(3 * rand('uniform'));
        si = 1 + floor(2 * rand('uniform'));

        ΕΑΝ fi = 1 ΤΟΤΕ facility = 'NorthClinic';
        ΑΛΛΙΩΣ ΕΑΝ fi = 2 ΤΟΤΕ facility = 'CentralMed';
        ΑΛΛΙΩΣ facility = 'SouthClinic';
        ΕΑΝ si = 1 ΤΟΤΕ specialty = 'Cardiology';
        ΑΛΛΙΩΣ specialty = 'Oncology';

        /* Γνήσιος όρος αλληλεπίδρασης εγκατάστασης x ειδικότητας */
        interaction = 1.2 * f_bias[fi] * s_bias[si];

        satisfaction = 7.0 + f_bias[fi] + s_bias[si] + interaction
            + rand('normal', 0, 0.25);

        /* Διατήρηση σε κλίμακα εμπειρίας ασθενή 1-10 */
        ΕΑΝ satisfaction > 10 ΤΟΤΕ satisfaction = 10;
        ΕΑΝ satisfaction < 1  ΤΟΤΕ satisfaction = 1;
        satisfaction = round(satisfaction, 0.1);
        ΕΞΟΔΟΣ;
    ΤΕΛΟΣ;

    ΚΡΑΤΗΣΗ facility specialty satisfaction;
ΕΚΤΕΛΕΣΗ;



NOTE: DATA encounters


NOTE: Wrote encounters (100 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Βήμα 2 - Επιθεώρηση της ακατέργαστης κατανομής βαθμολογιών

Πριν τη μοντελοποίηση, επιβεβαιώνουμε ότι οι συνθετικές βαθμολογίες συμπεριφέρονται καλά και εξετάζουμε τους μέσους όρους ανά κελί εγκατάστασης x ειδικότητας. Οι λέξεις-κλειδιά εκατοστημορίων της `PROC MEANS` (`P25`, `MEDIAN`, `P75`) συνοψίζουν τη συνολική διασπορά· η δεύτερη κλήση δείχνει τους μέσους όρους ανά κελί που φέρουν την αλληλεπίδραση που η μηχανή παραγοντοποίησης θα προσπαθήσει να ανακτήσει.

In [2]:
ΔΙΑΔΙΚΑΣΙΑ ΜΕΣΟΙ ΔΕΔΟΜΕΝΑ=encounters n mean std MIN p25 MEDIAN p75 MAX maxdec=2;
    ΜΕΤΑΒΛΗΤΗ satisfaction;
    ΕΤΙΚΕΤΑ satisfaction="Ικανοποίηση";
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ ΜΕΣΟΙ ΔΕΔΟΜΕΝΑ=encounters mean NWAY maxdec=2;
    ΚΛΑΣΗ facility specialty;
    ΜΕΤΑΒΛΗΤΗ satisfaction;
    ΕΤΙΚΕΤΑ facility="Εγκατάσταση" specialty="Ειδικότητα" satisfaction="Ικανοποίηση";
ΕΚΤΕΛΕΣΗ;


                                                  The MEANS Procedure

 Variable      Label                          N        Mean     Std Dev     Minimum   Lower Quartile      Median   Upper Quartile     Maximum
 --------------------------------------------------------------------------------------------------------------------------------------------
 satisfaction  Ικανοποίηση                  100        6.75        1.81        4.40             5.60        6.20             8.00       10.00
 --------------------------------------------------------------------------------------------------------------------------------------------

                                                  The MEANS Procedure

                                Analysis Variable : satisfaction Ικανοποίηση

                                                                                N
                            Εγκατάσταση               Ειδικότητα              Obs       Mean
                            --------


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Βήμα 3 - Προσαρμογή της μηχανής παραγοντοποίησης

Μοντελοποιούμε το `satisfaction` ως τον **στόχο** διαστήματος και τα δύο κατηγορικά πεδία `facility` και `specialty` ως ονομαστικά χαρακτηριστικά **εισόδου**. Βασικές επιλογές:

- `LEARN=0.02` - το μέγεθος βήματος της στοχαστικής καθόδου κλίσης. Σε αυτόν τον μικρό, τυποποιημένο σχεδιασμό, ένας μέτριος ρυθμός διατηρεί τον βελτιστοποιητή σταθερό ενώ εξακολουθεί να προσαρμόζει τη δομή κελιού.
- `L2=0.0005` - ελαφριά κανονικοποίηση L2, αρκετή ώστε να εμποδίσει τους παράγοντες από το να συρρικνώνονται υπερβολικά προς τον γενικό μέσο όρο.
- `SEED=20240531` - αναπαραγώγιμη αρχικοποίηση.
- `OUT=scored` - εγγραφή προβλέψεων ανά γραμμή (`P_satisfaction`).
- `OUTSTAT=fitstats` - καταγραφή του ιστορικού RMSE ανά επανάληψη ώστε να επιβεβαιώσουμε τη σύγκλιση.

Η δήλωση `ID` αντιγράφει τα βασικά πεδία στην έξοδο βαθμολόγησης ώστε κάθε πρόβλεψη να παραμένει επισημασμένη με την εγκατάστασή και την ειδικότητά της.

In [3]:
ΔΙΑΔΙΚΑΣΙΑ FACTMAC ΔΕΔΟΜΕΝΑ=encounters seed=20240531 learn=0.02 l2=0.0005
        out=scored OUTSTAT=fitstats;
    ΕΙΣΟΔΟΣ facility specialty / level=nominal;
    target satisfaction / level=interval;
    id facility specialty;
    ΕΤΙΚΕΤΑ facility="Εγκατάσταση" specialty="Ειδικότητα" satisfaction="Ικανοποίηση";
ΕΚΤΕΛΕΣΗ;



                         The FACTMAC Procedure

  Target variable: Ικανοποίηση
  Input variable: Εγκατάσταση
  Input variable: Ειδικότητα

Fit Statistics

  L2 Regularization              0.0005
  Learning Rate                  0.02
  Max Iterations                 100
  Number of Factors              10
  Number of Features             2
  Number of Observations         100
  Train ASE                      1.561623
  Train MAE                      0.953557
  Train R-Square                 0.515847
  Train RASE                     1.249649

Variable Importance

  Variable                            Importance
  --------                            ----------
  SPECIALTY                             0.513485
  FACILITY                              0.486515




NOTE: PROC FACTMAC data=encounters

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC FACTMAC completed.


## Βήμα 4 - Επιβεβαίωση σύγκλισης

Ο πίνακας `OUTSTAT=` καταγράφει το RMSE εκπαίδευσης σε κάθε επανάληψη SGD. Μια μονοτονικά φθίνουσα καμπύλη που ισοπεδώνεται μας λέει ότι το μοντέλο έχει συγκλίνει εντός του (προεπιλεγμένου 100) προϋπολογισμού επαναλήψεων.

In [4]:
ΔΙΑΔΙΚΑΣΙΑ ΕΚΤΥΠΩΣΗ ΔΕΔΟΜΕΝΑ=fitstats(obs=15) ΕΤΙΚΕΤΑ noobs;
ΕΚΤΕΛΕΣΗ;



ITERATION          RMSE
---------  ------------
1          1.6784734207
2          1.2915242083
3          1.2679925124
4          1.2654232966
5          1.2650259551
6          1.2649577538
7          1.2649457032
8          1.2649435534
9          1.2649431684
10         1.2649430993
11         1.2649430869
12         1.2649430847
13         1.2649430843
14         1.2649430842
15         1.2649430842

... 85 more observations (showing 15 of 100)




NOTE: PROC PRINT data=fitstats

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 15 observations printed, 2 variables


## Βήμα 5 - Αξιολόγηση σφάλματος ανακατασκευής

Ο πίνακας βαθμολόγησης φέρει το παρατηρούμενο `satisfaction` δίπλα στο `P_satisfaction` του μοντέλου. Παράγουμε το υπόλοιπο και το απόλυτο σφάλμα, και έπειτα συνοψίζουμε. Ένα υπόλοιπο κεντραρισμένο κοντά στο μηδέν και μια διασπορά προβλεπόμενης βαθμολογίας που πλησιάζει την παρατηρούμενη διασπορά (αντί να συρρικνώνεται σε μια επίπεδη γραμμή στον γενικό μέσο όρο) υποδεικνύουν ότι η μηχανή παραγοντοποίησης αναπαράγει την ενσωματωμένη δομή εγκατάστασης x ειδικότητας.

In [5]:
ΔΕΔΟΜΕΝΑ resid;
    ΟΡΙΣΜΟΣ scored;
    residual = satisfaction - p_satisfaction;
    abs_err  = abs(residual);
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ ΕΚΤΥΠΩΣΗ ΔΕΔΟΜΕΝΑ=scored(obs=10) ΕΤΙΚΕΤΑ noobs;
    ΕΤΙΚΕΤΑ facility="Εγκατάσταση" specialty="Ειδικότητα" satisfaction="Ικανοποίηση"
          p_satisfaction="Προβλεπόμενη Ικανοποίηση";
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ ΜΕΣΟΙ ΔΕΔΟΜΕΝΑ=resid n mean std MIN p25 MEDIAN p75 MAX maxdec=3;
    ΜΕΤΑΒΛΗΤΗ satisfaction p_satisfaction residual abs_err;
    ΕΤΙΚΕΤΑ satisfaction="Ικανοποίηση" p_satisfaction="Προβλεπόμενη Ικανοποίηση"
          residual="Υπόλοιπο" abs_err="Απόλυτο Σφάλμα";
ΕΚΤΕΛΕΣΗ;


           Εγκατάσταση            Ειδικότητα             Ικανοποίηση                         Προβλεπόμενη Ικανοποίηση
----------------------  --------------------  ----------------------  -----------------------------------------------
SouthClinic             Oncology                                 6.3                                     6.1577265381
NorthClinic             Oncology                                 5.4                                     6.0430846516
CentralMed              Cardiology                               7.9                                     9.5419769923
SouthClinic             Cardiology                               4.7                                     5.8019161993
CentralMed              Oncology                                 6.2                                     5.9284427651
NorthClinic             Cardiology                                10                                     7.6719465958
NorthClinic             Oncology                        


NOTE: DATA resid


NOTE: Read 100 rows from scored.
NOTE: Wrote resid (100 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC PRINT data=scored

NOTE: PROC PRINT completed: 10 observations printed, 4 variables
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Βήμα 6 - Ανάδειξη απόδοσης εγκατάστασης x ειδικότητας

Για ομάδες βελτίωσης ποιότητας, η αξιοποιήσιμη προβολή είναι η προβλεπόμενη βαθμολογία ανά **συνδυασμό εγκατάστασης x ειδικότητας**. Οι συνδυασμοί των οποίων η προβλεπόμενη από το μοντέλο εμπειρία βρίσκεται αρκετά κάτω από τον μέσο όρο του συστήματος είναι υποψήφιοι για αναθεώρηση· η στήλη απόλυτου σφάλματος δείχνει πού το μοντέλο προσαρμόζεται καθαρά και πού το περικομμένο όριο 1-10 περιορίζει πόσο ψηλά μπορεί να φτάσει μια γραμμική μηχανή παραγοντοποίησης.

In [6]:
ΔΙΑΔΙΚΑΣΙΑ ΜΕΣΟΙ ΔΕΔΟΜΕΝΑ=resid mean NWAY maxdec=3;
    ΚΛΑΣΗ facility specialty;
    ΜΕΤΑΒΛΗΤΗ p_satisfaction abs_err;
    ΕΤΙΚΕΤΑ facility="Εγκατάσταση" specialty="Ειδικότητα"
          p_satisfaction="Προβλεπόμενη Ικανοποίηση" abs_err="Απόλυτο Σφάλμα";
ΕΚΤΕΛΕΣΗ;


                                                  The MEANS Procedure

                            Analysis Variable : p_satisfaction Προβλεπόμενη Ικανοποίηση

                                                                                N
                            Εγκατάσταση               Ειδικότητα              Obs       Mean
                            ----------------------------------------------------------------
                            CentralMed                Cardiology               13      9.542

                                                      Oncology                 20      5.928

                            NorthClinic               Cardiology               18      7.672

                                                      Oncology                 16      6.043

                            SouthClinic               Cardiology               20      5.802

                                                      Oncology                 13      6.158
         


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Ερμηνεία των Αποτελεσμάτων

Το ιστορικό επαναλήψεων από το `OUTSTAT=` δείχνει το RMSE εκπαίδευσης να μειώνεται από περίπου 1.68 στο πρώτο πέρασμα σε ένα οροπέδιο κοντά στο **1.265** γύρω στην έβδομη επανάληψη, επιβεβαιώνοντας ότι η προσαρμογή SGD συνέκλινε καλά εντός του προϋπολογισμού επαναλήψεων. Οι Στατιστικές Προσαρμογής αναφέρουν ένα **R-Square εκπαίδευσης 0.516**, ένα **μέσο απόλυτο σφάλμα 0.954** μονάδες βαθμολογίας, και ένα **RASE 1.25** — η μηχανή παραγοντοποίησης εξηγεί περίπου τον μισό της διακύμανσης σε μια βαθμολογία ικανοποίησης 1-10 που έχει τυπική απόκλιση 1.81, οπότε πράγματι μαθαίνει δομή αντί να προβλέπει τον γενικό μέσο όρο. Η σύνοψη υπολοίπων το επιβεβαιώνει: ο μέσος όρος υπολοίπου είναι ουσιαστικά μηδέν (0.020) και οι προβλεπόμενες βαθμολογίες εκτείνονται από 5.80 έως 9.54 (τυπική απόκλιση 1.27), ακολουθώντας — αν και όχι πλήρως αντιστοιχώντας — την παρατηρούμενη διασπορά.

Ο πίνακας εγκατάστασης x ειδικότητας μετατρέπει το λανθάνον μοντέλο σε κάτι που μια ομάδα ποιότητας περίθαλψης μπορεί να αξιοποιήσει. Το μοντέλο κατατάσσει το `CentralMed`/`Cardiology` ψηλότερα (προβλεπόμενο 9.54) και το `SouthClinic`/`Cardiology` χαμηλότερα (προβλεπόμενο 5.80), ανακτώντας την ενσωματωμένη αλληλεπίδραση στην οποία η Καρδιολογία είναι εξαιρετική σε ορισμένες εγκαταστάσεις και ασθενής σε άλλες. Η στήλη απόλυτου σφάλματος είναι ειλικρινής σχετικά με τα όρια του μοντέλου: τα δύο κελιά Ογκολογίας αναπαράγονται σχεδόν ακριβώς (μέσο απόλυτο σφάλμα 0.19-0.24), ενώ το `NorthClinic`/`Cardiology` υπο-προβλέπεται (σφάλμα 2.33) επειδή οι πραγματικές του βαθμολογίες συσσωρεύονται στο περικομμένο όριο 1-10 που μια γραμμική μηχανή παραγοντοποίησης δεν μπορεί να φτάσει.

**Επόμενα βήματα** που θα μπορούσε να κάνει ένας επαγγελματίας: προσθήκη ενός `PARTITION` κρατημένου συνόλου για προστασία από υπερπροσαρμογή, συντονισμός των `LEARN=` και `L2=` για ανταλλαγή μεροληψίας με διακύμανση, ή επέκταση του συνόλου χαρακτηριστικών (πάροχος, τύπος επίσκεψης, εποχή) ώστε η μηχανή παραγοντοποίησης να μπορεί να μοντελοποιήσει οδηγούς εμπειρίας υψηλότερης τάξης. Σε μια πλήρως αδειοδοτημένη εγκατάσταση, ένα μεγαλύτερο πλέγμα εγκατάστασης x ειδικότητας με περισσότερες επισκέψεις ανά κελί θα επέτρεπε στο μοντέλο να επιλύσει λεπτότερη δομή αλληλεπίδρασης από τον σχεδιασμό έξι κελιών που παρουσιάζεται εδώ.